In [28]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 39.4 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 MB 32.1 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import xgboost as xgb

In [2]:
df = pd.read_csv('euroncap_cardata_masked.csv')
df = df[~df['stars'].isna()]
df = df.drop(columns=['type'])

In [3]:
df.columns

Index(['id', 'make', 'model', 'year', 'stars', 'drive_hand', 'protocol_year',
       'kerb_weight', 'seat_arrangement', 'rating_year',
       'seat1_frontal_airbag', 'seat3_frontal_airbag', 'row2_frontal_airbag',
       'seat1_pretensioner', 'seat3_pretensioner', 'row2_pretensioner',
       'seat1_load_limiter', 'seat3_load_limiter', 'row2_load_limiter',
       'seat1_knee_airbag', 'seat3_knee_airbag', 'seat1_side_head_airbag',
       'seat3_side_head_airbag', 'row2_side_head_airbag',
       'seat1_side_chest_airbag', 'seat3_side_chest_airbag',
       'row2_side_chest_airbag', 'seat1_side_pelvis_airbag',
       'seat3_side_pelvis_airbag', 'row2_side_pelvis_airbag',
       'seat1_centre_lateral_airbag', 'seat3_centre_lateral_airbag',
       'row2_centre_lateral_airbag', 'seat1_isofix', 'seat3_isofix',
       'row2_isofix', 'row3_isofix', 'seat1_isize', 'seat3_isize',
       'row2_isize', 'row3_isize', 'seat1_integrated_child_seat',
       'seat3_integrated_child_seat', 'row2_integrated_

In [4]:
to_be_dropped = ['id', 'make', 'model','year', 'stars', 'protocol_year', 'seat_arrangement', 'rating_year', 'seat1_frontal_airbag', 'seat3_frontal_airbag', 'row2_frontal_airbag', 'seat1_pretensioner', 'seat3_pretensioner', 'row2_pretensioner', 'seat1_load_limiter', 'seat3_load_limiter', 'row2_load_limiter', 'seat3_knee_airbag', 'seat1_side_head_airbag', 'seat3_side_head_airbag', 'row2_side_head_airbag', 'seat1_side_chest_airbag', 'seat3_side_chest_airbag', 'row2_centre_lateral_airbag', 'seat1_isofix', 'row2_isofix', 'row3_isofix', 'seat1_integrated_child_seat','seat3_integrated_child_seat', 'row2_integrated_child_seat','row3_integrated_child_seat', 'seat1_airbagcutoffswitch','seat3_airbagcutoffswitch', 'row2_airbagcutoffswitch','row3_airbagcutoffswitch','seat1_childdetection','row3_childdetection']
features = ['kerb_weight', 'seat1_knee_airbag', 'row2_side_chest_airbag', 'seat1_side_pelvis_airbag','seat3_side_pelvis_airbag', 'row2_side_pelvis_airbag', 'seat1_centre_lateral_airbag', 'seat3_centre_lateral_airbag', 'seat3_isofix', 'seat1_isize', 'seat3_isize','row2_isize', 'row3_isize', 'seat3_childdetection', 'row2_childdetection','has_active_bonnet','has_distraction_detection', 'has_fatigue_detection', 'has_lane_assist', 'has_speed_assist', 'has_aeb_vru', 'has_cyclistdoorprevention','has_aeb_m2w', 'has_aeb_cartocar']
# df[[]].value_counts()

In [5]:
len(features)

24

In [6]:
len(to_be_dropped) + len(features)

61

In [7]:
df.shape

(459, 62)

In [8]:
df['stars'].value_counts().sort_index()

stars
1.0      3
2.0      4
3.0     32
4.0     89
5.0    331
Name: count, dtype: int64

In [9]:
star_map = {1: 'low', 2:'low', 3:'low', 4:'medium', 5:'high'}
df['stars_category'] = df['stars'].map(star_map)
df.columns

Index(['id', 'make', 'model', 'year', 'stars', 'drive_hand', 'protocol_year',
       'kerb_weight', 'seat_arrangement', 'rating_year',
       'seat1_frontal_airbag', 'seat3_frontal_airbag', 'row2_frontal_airbag',
       'seat1_pretensioner', 'seat3_pretensioner', 'row2_pretensioner',
       'seat1_load_limiter', 'seat3_load_limiter', 'row2_load_limiter',
       'seat1_knee_airbag', 'seat3_knee_airbag', 'seat1_side_head_airbag',
       'seat3_side_head_airbag', 'row2_side_head_airbag',
       'seat1_side_chest_airbag', 'seat3_side_chest_airbag',
       'row2_side_chest_airbag', 'seat1_side_pelvis_airbag',
       'seat3_side_pelvis_airbag', 'row2_side_pelvis_airbag',
       'seat1_centre_lateral_airbag', 'seat3_centre_lateral_airbag',
       'row2_centre_lateral_airbag', 'seat1_isofix', 'seat3_isofix',
       'row2_isofix', 'row3_isofix', 'seat1_isize', 'seat3_isize',
       'row2_isize', 'row3_isize', 'seat1_integrated_child_seat',
       'seat3_integrated_child_seat', 'row2_integrated_

In [10]:
df['stars_category'].value_counts()

stars_category
high      331
medium     89
low        39
Name: count, dtype: int64

In [11]:
def map_protocol(year):
    if year <= 2019:
        return 1
    elif year <= 2021:  
        return 2
    elif year <= 2025:
        return 3
    
df['protocol_version'] = df['protocol_year'].apply(map_protocol)

In [12]:
features.append('protocol_version')
# features.append('stars_category')
features

['kerb_weight',
 'seat1_knee_airbag',
 'row2_side_chest_airbag',
 'seat1_side_pelvis_airbag',
 'seat3_side_pelvis_airbag',
 'row2_side_pelvis_airbag',
 'seat1_centre_lateral_airbag',
 'seat3_centre_lateral_airbag',
 'seat3_isofix',
 'seat1_isize',
 'seat3_isize',
 'row2_isize',
 'row3_isize',
 'seat3_childdetection',
 'row2_childdetection',
 'has_active_bonnet',
 'has_distraction_detection',
 'has_fatigue_detection',
 'has_lane_assist',
 'has_speed_assist',
 'has_aeb_vru',
 'has_cyclistdoorprevention',
 'has_aeb_m2w',
 'has_aeb_cartocar',
 'protocol_version']

In [13]:
x = df[features]
y = df['stars_category']

X_train, X_test, y_train, y_test = train_test_split(
    x, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y  # Keeps the 80/20 ratio in both sets
)

In [14]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(367, 25) (92, 25) (367,) (92,)


In [15]:
print(y_train.value_counts(normalize=True))

stars_category
high      0.722071
medium    0.193460
low       0.084469
Name: proportion, dtype: float64


In [16]:
print(y_test.value_counts(normalize=True))

stars_category
high      0.717391
medium    0.195652
low       0.086957
Name: proportion, dtype: float64


## Random_forest - Class weight unbalanced

In [17]:
model = RandomForestClassifier(n_estimators=100, random_state=42)

# 5. Train the model on training data
model.fit(X_train, y_train)

# 6. Predict classes for the testing dataset
y_pred = model.predict(X_test)

# 7. Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=df['stars_category'].unique()))

# Optional: View feature importance rankings
importances = pd.Series(model.feature_importances_, index=df[features].columns)
print("Feature Importances:")
print(importances.sort_values(ascending=False))

Model Accuracy: 82.61%

Classification Report:
              precision    recall  f1-score   support

        high       0.84      0.97      0.90        66
         low       0.50      0.25      0.33         8
      medium       0.83      0.56      0.67        18

    accuracy                           0.83        92
   macro avg       0.73      0.59      0.63        92
weighted avg       0.81      0.83      0.81        92

Feature Importances:
kerb_weight                    0.360392
seat1_centre_lateral_airbag    0.080124
has_lane_assist                0.063946
has_active_bonnet              0.036630
has_fatigue_detection          0.035493
has_aeb_m2w                    0.034227
seat1_side_pelvis_airbag       0.031954
has_cyclistdoorprevention      0.031498
seat3_side_pelvis_airbag       0.031269
protocol_version               0.031255
seat1_knee_airbag              0.031003
seat3_isofix                   0.027962
seat3_isize                    0.024984
seat3_centre_lateral_airbag    

## Retrain Model - Balanced Class weight

In [63]:
model = RandomForestClassifier(max_depth=10, min_samples_split=5, n_estimators=100, random_state=42, class_weight='balanced')

# 5. Train the model on training data
model.fit(X_train, y_train)

# 6. Predict classes for the testing dataset
y_pred = model.predict(X_test)

# 7. Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=df['stars_category'].unique()))

# Optional: View feature importance rankings
importances = pd.Series(model.feature_importances_, index=df[features].columns)
print("Feature Importances:")
print(importances.sort_values(ascending=False))

Model Accuracy: 83.70%

Classification Report:
              precision    recall  f1-score   support

        high       0.92      0.89      0.91        66
         low       0.50      0.62      0.56         8
      medium       0.72      0.72      0.72        18

    accuracy                           0.84        92
   macro avg       0.71      0.75      0.73        92
weighted avg       0.85      0.84      0.84        92

Feature Importances:
kerb_weight                    0.258254
has_lane_assist                0.136252
seat1_centre_lateral_airbag    0.084079
seat1_side_pelvis_airbag       0.041554
seat3_side_pelvis_airbag       0.038819
seat3_isofix                   0.037797
has_active_bonnet              0.035886
has_aeb_vru                    0.034284
has_cyclistdoorprevention      0.029777
protocol_version               0.029735
seat3_isize                    0.027895
seat1_knee_airbag              0.026636
has_aeb_m2w                    0.025379
has_speed_assist               

In [64]:
import joblib

# Save the trained model to a file
joblib.dump(model, 'euro_ncap_model.joblib')

# Load the saved model back into memory
loaded_model = joblib.load('euro_ncap_model.joblib')


In [56]:
model = RandomForestClassifier(max_depth=10, min_samples_split=2, n_estimators=300, random_state=42, class_weight='balanced')

# 5. Train the model on training data
model.fit(X_train, y_train)

# 6. Predict classes for the testing dataset
y_pred = model.predict(X_test)

# 7. Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=df['stars_category'].unique()))

# Optional: View feature importance rankings
importances = pd.Series(model.feature_importances_, index=df[features].columns)
print("Feature Importances:")
print(importances.sort_values(ascending=False))

Model Accuracy: 82.61%

Classification Report:
              precision    recall  f1-score   support

        high       0.90      0.91      0.90        66
         low       0.56      0.62      0.59         8
      medium       0.69      0.61      0.65        18

    accuracy                           0.83        92
   macro avg       0.71      0.72      0.71        92
weighted avg       0.83      0.83      0.83        92

Feature Importances:
kerb_weight                    0.298218
has_lane_assist                0.115869
seat1_centre_lateral_airbag    0.071179
seat3_side_pelvis_airbag       0.039721
has_active_bonnet              0.038289
seat1_side_pelvis_airbag       0.035259
has_aeb_vru                    0.034929
seat1_knee_airbag              0.034622
seat3_isofix                   0.032274
seat3_isize                    0.031272
has_cyclistdoorprevention      0.029004
protocol_version               0.028746
has_speed_assist               0.025756
has_aeb_m2w                    

### hyperparameter tuning

In [59]:
params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced', None]
}

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

best_model = GridSearchCV(estimator=RandomForestClassifier(random_state=42), param_grid=params, cv=5, n_jobs=-1, verbose=1, scoring='f1_macro')
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("Best Params:", best_model.best_params_)
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Params: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
              precision    recall  f1-score   support

        high       0.92      0.89      0.91        66
         low       0.50      0.62      0.56         8
      medium       0.72      0.72      0.72        18

    accuracy                           0.84        92
   macro avg       0.71      0.75      0.73        92
weighted avg       0.85      0.84      0.84        92



## random forest - using smote

In [27]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
y_train_resampled.value_counts()

stars_category
high      265
medium    265
low       265
Name: count, dtype: int64

In [26]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
# 5. Train the model on training data
model.fit(X_train_resampled, y_train_resampled)

# 6. Predict classes for the testing dataset
y_pred = model.predict(X_test)

# 7. Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=df['stars_category'].unique()))

# Optional: View feature importance rankings
importances = pd.Series(model.feature_importances_, index=df[features].columns)
print("Feature Importances:")
print(importances.sort_values(ascending=False))

Model Accuracy: 80.43%

Classification Report:
              precision    recall  f1-score   support

        high       0.85      0.92      0.88        66
         low       0.60      0.38      0.46         8
      medium       0.67      0.56      0.61        18

    accuracy                           0.80        92
   macro avg       0.70      0.62      0.65        92
weighted avg       0.79      0.80      0.79        92

Feature Importances:
kerb_weight                    0.166886
has_lane_assist                0.160823
seat1_centre_lateral_airbag    0.087538
has_aeb_vru                    0.063841
has_distraction_detection      0.051926
has_aeb_cartocar               0.045677
seat1_side_pelvis_airbag       0.044567
seat3_side_pelvis_airbag       0.044206
has_aeb_m2w                    0.030991
seat3_isofix                   0.030161
has_cyclistdoorprevention      0.029333
has_fatigue_detection          0.028439
has_active_bonnet              0.028434
seat1_knee_airbag              

## XGboost

In [31]:
star_map = {'low': 0, 'medium': 1, 'high': 2}
y_train_mapped = y_train.map(star_map)
y_test_mapped = y_test.map(star_map)

In [33]:
clf = xgb.XGBClassifier(
    n_estimators=100,      # Number of boosting rounds (trees)
    learning_rate=0.1,     # Step size shrinkage to prevent overfitting
    max_depth=5,           # Maximum depth of a tree
    tree_method="hist",    # Use histogram-based algorithm for speed
    random_state=42
)

# 5. Fit the model to the training data
clf.fit(X_train, y_train_mapped)

# 6. Predict class labels on the test data
y_pred = clf.predict(X_test)

# 7. Predict class probabilities (optional)
y_proba = clf.predict_proba(X_test)

# 8. Evaluate model performance
accuracy = accuracy_score(y_test_mapped, y_pred)
print(f"Test Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test_mapped, y_pred))


Test Accuracy: 0.8261

Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.25      0.29         8
           1       0.85      0.61      0.71        18
           2       0.86      0.95      0.91        66

    accuracy                           0.83        92
   macro avg       0.68      0.61      0.63        92
weighted avg       0.81      0.83      0.81        92



## xgboost with class balance

In [34]:
star_map = {'low': 0, 'medium': 1, 'high': 2}
y_train_mapped = y_train.map(star_map)
y_test_mapped = y_test.map(star_map)

In [36]:
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight

# 1. Initialize your multiclass XGBoost model
model = xgb.XGBClassifier(objective='multi:softprob', num_class=3)

# 2. Compute a weight vector matching the length of your training labels (y_train)
# 'balanced' automatically calculates weights inversely proportional to class frequencies
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_mapped)

# 3. Pass the sample weights array directly into the fit method
model.fit(X_train, y_train_mapped, sample_weight=sample_weights)

# 8. Evaluate model performance
accuracy = accuracy_score(y_test_mapped, y_pred)
print(f"Test Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test_mapped, y_pred))

Test Accuracy: 0.8261

Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.25      0.29         8
           1       0.85      0.61      0.71        18
           2       0.86      0.95      0.91        66

    accuracy                           0.83        92
   macro avg       0.68      0.61      0.63        92
weighted avg       0.81      0.83      0.81        92



## Cross validation

In [51]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'), df[features], df['stars_category'], cv=cv, scoring='f1_macro')

print(scores)
print(f'Mean F1: {scores.mean():.4f} (+- {scores.std():.4f})')

[0.62525318 0.64664987 0.57696477 0.6792328  0.61576591]
Mean F1: 0.6288 (+- 0.0339)
